In [ ]:
# Import libraries for data preprocessing
import numpy as np
from sklearn.datasets import load_breast_cancer

# Import libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.optim import SGD

## Classification

#### Data Preprocessing

In [11]:
# Loading dataset
data = load_breast_cancer()

# Split into x and y
x = data.data
y = data.target

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

(455, 30)
(114, 30)
(455,)
(114,)


#### Baseline

In [ ]:
# Logistic Regression using scikit-learn
baseline = LogisticRegression(max_iter=10000)

# Train the model
baseline.fit(X_train, y_train)

# Predict y-values
baseline_pred = baseline.predict(X_test)

# Calculate accuracy score
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline logistic regression accuracy: {baseline_acc:.4f}") 

Baseline logistic regression accuracy: 0.9561


#### Logistic Regression using PyTorch

In [35]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(input_dim, 1))
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, X):
        z = X @ self.w + self.b
        return torch.sigmoid(z)

# Custom Binary Cross-Entropy Loss with Clipping
def custom_bce_loss(y_pred, y_true):
    # Clip to prevent log(0) numerical instability
    y_pred = torch.clamp(y_pred, 1e-7, 1.0 - 1e-7)
    return -torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))

# Train Logistic Regression
lr_model = LogisticRegressionModel(X_train.shape[1])
optimizer_lr = SGD(lr_model.parameters(), lr=1e-5)
    
for epoch in range(1000):
    optimizer_lr.zero_grad()
    predictions = lr_model(X_train_t)
    loss = custom_bce_loss(predictions, y_train_t)
    loss.backward()
    optimizer_lr.step()

In [42]:
# Evaluate Custom Logistic Regression
with torch.no_grad():
    lr_test_preds = (lr_model(X_test_t) >= 0.5).float()
    lr_acc = (lr_test_preds == y_test_t).float().mean().item()

print("\nTest Accuracy Comparison")
print(f"Baseline Logistic Regression:     {baseline_acc:.4f}")
print(f"Implemented Logistic Regression:  {lr_acc:.4f}")


Test Accuracy Comparison
Baseline Logistic Regression:     0.9561
Implemented Logistic Regression:  0.9474


The performance seems very similar to baseline

#### Probit Regression

In [39]:
class ProbitRegressionCustom(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(input_dim, 1))
        self.b = nn.Parameter(torch.zeros(1))
        self.normal_dist = Normal(0, 1) # Standard Normal
        
    def forward(self, X):
        z = X @ self.w + self.b
        # Φ(z) represents the Standard Normal Cumulative Distribution Function (CDF)
        return self.normal_dist.cdf(z)

# Train Probit Regression
probit_model = ProbitRegressionCustom(X_train.shape[1])
optimizer_probit = SGD(probit_model.parameters(), lr=1e-7)

for epoch in range(1000):
    optimizer_probit.zero_grad()
    predictions = probit_model(X_train_t)
    loss = custom_bce_loss(predictions, y_train_t)
    loss.backward()
    optimizer_probit.step()

In [44]:
# Evaluate Custom Probit Regression
with torch.no_grad():
    probit_test_preds = (probit_model(X_test_t) >= 0.5).float()
    probit_acc = (probit_test_preds == y_test_t).float().mean().item()

print("\nTest Accuracy Comparison")
print(f"Baseline Logistic Regression:     {baseline_acc:.4f}")
print(f"Implemented Probit Regression:    {probit_acc:.4f}")


Test Accuracy Comparison
Baseline Logistic Regression:     0.9561
Implemented Probit Regression:    0.9298


The performance seems very similar to baseline